# Parallel Image Filters: Serial vs OpenMP vs CUDA

Gaussian blur + Sobel edge detection on HD → 8K images, timed on one CPU core,
on all CPU cores (OpenMP) and on the GPU (CUDA).

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

## 1. Check the hardware

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!echo "CPU cores: $(nproc)"

## 2. Write the source files
(Same files as in the project folder.)

In [ ]:
%%writefile common.h
/*
 * common.h - shared helpers for all three versions (serial, OpenMP, CUDA).
 *
 *   - read/write grayscale PGM images (the simplest image format: a tiny text
 *     header followed by raw bytes, one byte per pixel)
 *   - a millisecond timer
 *   - the per-pixel filter math used by the serial and OpenMP versions
 *
 * It compiles as C (gcc/clang) and as C++ (nvcc), so the CUDA version can
 * reuse the image I/O and the edge-strength formula.
 */
#ifndef COMMON_H
#define COMMON_H

#include <ctype.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

/* Functions marked HD run on both the CPU and the GPU when compiled by nvcc. */
#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

typedef struct {
    int width;
    int height;
    uint8_t *data; /* width * height bytes, row by row */
} Image;

static inline void die(const char *msg) {
    fprintf(stderr, "error: %s\n", msg);
    exit(1);
}

static inline void *xmalloc(size_t n) {
    void *p = malloc(n);
    if (!p) die("out of memory");
    return p;
}

/* ---------------------------------------------------------------- PGM I/O */

static inline void pgm_skip(FILE *f) { /* skip whitespace and # comments */
    int c;
    while ((c = fgetc(f)) != EOF) {
        if (c == '#') {
            while ((c = fgetc(f)) != EOF && c != '\n') {}
        } else if (!isspace(c)) {
            ungetc(c, f);
            return;
        }
    }
}

static inline int pgm_int(FILE *f) {
    int v;
    pgm_skip(f);
    if (fscanf(f, "%d", &v) != 1) die("malformed PGM header");
    return v;
}

static inline Image read_pgm(const char *path) {
    FILE *f = fopen(path, "rb");
    if (!f) { perror(path); exit(1); }

    char magic[3] = {0};
    if (fread(magic, 1, 2, f) != 2 || strcmp(magic, "P5") != 0)
        die("input must be a binary grayscale PGM (P5) - use: python3 tools.py prepare");

    Image img;
    img.width = pgm_int(f);
    img.height = pgm_int(f);
    int maxval = pgm_int(f);
    if (img.width <= 0 || img.height <= 0 || maxval != 255)
        die("unsupported PGM (need 8-bit grayscale)");
    fgetc(f); /* exactly one whitespace byte separates header and pixels */

    size_t n = (size_t)img.width * img.height;
    img.data = (uint8_t *)xmalloc(n);
    if (fread(img.data, 1, n, f) != n) die("PGM file is truncated");
    fclose(f);
    return img;
}

static inline void write_pgm(const char *path, const uint8_t *data, int w, int h) {
    FILE *f = fopen(path, "wb");
    if (!f) { perror(path); exit(1); }
    fprintf(f, "P5\n%d %d\n255\n", w, h);
    fwrite(data, 1, (size_t)w * h, f);
    fclose(f);
}

/* ------------------------------------------------------------ timing / CSV */

static inline double now_ms(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec * 1e3 + ts.tv_nsec / 1e6;
}

/* Every program prints exactly one CSV line to stdout; bench.sh adds the header:
 * version,threads,width,height,blur_ms,sobel_ms,transfer_ms,total_ms
 * (for CUDA, "threads" = threads per block and transfer_ms = CPU<->GPU copies) */
static inline void print_result(const char *version, int threads, int w, int h,
                                double blur_ms, double sobel_ms, double transfer_ms) {
    printf("%s,%d,%d,%d,%.3f,%.3f,%.3f,%.3f\n", version, threads, w, h,
           blur_ms, sobel_ms, transfer_ms, blur_ms + sobel_ms + transfer_ms);
    fflush(stdout);
}

/* ------------------------------------------------------------ filter math */

static inline HD int clampi(int v, int lo, int hi) {
    return v < lo ? lo : (v > hi ? hi : v);
}

/* Edge strength from the Sobel gradients: sqrt(gx^2 + gy^2), capped at 255.
 * Shared with the GPU so every version produces bit-identical output. */
static inline HD uint8_t edge_value(int gx, int gy) {
    float mag = sqrtf((float)(gx * gx + gy * gy));
    return (uint8_t)(mag > 255.0f ? 255.0f : mag);
}

/* 5x5 Gaussian blur weights = [1 4 6 4 1] x [1 4 6 4 1]; they add up to 256. */
static const int GAUSS[5] = {1, 4, 6, 4, 1};

/* Blurred value of pixel (x, y). Pixels outside the image use the nearest
 * edge pixel ("clamp to edge"). */
static inline uint8_t blur_pixel(const uint8_t *in, int w, int h, int x, int y) {
    int sum = 0;
    for (int dy = -2; dy <= 2; dy++) {
        const uint8_t *row = in + (size_t)clampi(y + dy, 0, h - 1) * w;
        for (int dx = -2; dx <= 2; dx++)
            sum += GAUSS[dy + 2] * GAUSS[dx + 2] * row[clampi(x + dx, 0, w - 1)];
    }
    return (uint8_t)((sum + 128) >> 8); /* divide by 256, rounded */
}

/* Sobel edge strength of pixel (x, y) from its 3x3 neighbourhood:
 *     p00 p01 p02
 *     p10  .  p12
 *     p20 p21 p22                                                         */
static inline uint8_t sobel_pixel(const uint8_t *in, int w, int h, int x, int y) {
    const uint8_t *up = in + (size_t)clampi(y - 1, 0, h - 1) * w;
    const uint8_t *mid = in + (size_t)y * w;
    const uint8_t *down = in + (size_t)clampi(y + 1, 0, h - 1) * w;
    int xl = clampi(x - 1, 0, w - 1), xr = clampi(x + 1, 0, w - 1);

    int p00 = up[xl], p01 = up[x], p02 = up[xr];
    int p10 = mid[xl], p12 = mid[xr];
    int p20 = down[xl], p21 = down[x], p22 = down[xr];

    int gx = (p02 + 2 * p12 + p22) - (p00 + 2 * p10 + p20); /* left vs right */
    int gy = (p20 + 2 * p21 + p22) - (p00 + 2 * p01 + p02); /* top vs bottom */
    return edge_value(gx, gy);
}

#endif /* COMMON_H */

In [ ]:
%%writefile serial.c
/*
 * serial.c - the baseline: Gaussian blur, then Sobel edge detection,
 * one pixel at a time on a single CPU core.
 *
 * usage: ./serial input.pgm output.pgm [repeats]
 */
#include "common.h"

static void gaussian_blur(const uint8_t *in, uint8_t *out, int w, int h) {
    for (int y = 0; y < h; y++)
        for (int x = 0; x < w; x++)
            out[(size_t)y * w + x] = blur_pixel(in, w, h, x, y);
}

static void sobel(const uint8_t *in, uint8_t *out, int w, int h) {
    for (int y = 0; y < h; y++)
        for (int x = 0; x < w; x++)
            out[(size_t)y * w + x] = sobel_pixel(in, w, h, x, y);
}

int main(int argc, char **argv) {
    if (argc < 3) {
        fprintf(stderr, "usage: %s input.pgm output.pgm [repeats]\n", argv[0]);
        return 1;
    }
    int reps = argc > 3 ? atoi(argv[3]) : 5;
    if (reps < 1) reps = 1;

    Image img = read_pgm(argv[1]);
    int w = img.width, h = img.height;
    uint8_t *blurred = (uint8_t *)xmalloc((size_t)w * h);
    uint8_t *edges = (uint8_t *)xmalloc((size_t)w * h);

    /* Run several times and keep the fastest run (the first one also pays
     * for the OS handing us fresh memory pages). */
    double best_blur = 0, best_sobel = 0, best_total = 1e30;
    for (int r = 0; r < reps; r++) {
        double t0 = now_ms();
        gaussian_blur(img.data, blurred, w, h);
        double t1 = now_ms();
        sobel(blurred, edges, w, h);
        double t2 = now_ms();
        if (t2 - t0 < best_total) {
            best_total = t2 - t0;
            best_blur = t1 - t0;
            best_sobel = t2 - t1;
        }
    }

    write_pgm(argv[2], edges, w, h);
    print_result("serial", 1, w, h, best_blur, best_sobel, 0.0);

    free(img.data);
    free(blurred);
    free(edges);
    return 0;
}

In [ ]:
%%writefile openmp.c
/*
 * openmp.c - the same filters, with the image rows shared out among CPU threads.
 *
 * The only real change from serial.c is one line above each outer loop:
 *
 *     #pragma omp parallel for schedule(runtime)
 *
 * Variable scope: the loop counters y and x are private to each thread (every
 * thread has its own copy); the image pointers and sizes are shared (read-only,
 * so no race). Each thread writes different output pixels, so no locks are needed.
 *
 * schedule(runtime) lets us switch between static and dynamic scheduling with
 * the OMP_SCHEDULE environment variable, without recompiling.
 *
 * usage: OMP_NUM_THREADS=4 OMP_SCHEDULE=static ./openmp input.pgm output.pgm [repeats]
 */
#include "common.h"
#ifdef _OPENMP
#include <omp.h>
#endif

static void gaussian_blur(const uint8_t *in, uint8_t *out, int w, int h) {
    #pragma omp parallel for schedule(runtime)
    for (int y = 0; y < h; y++)
        for (int x = 0; x < w; x++)
            out[(size_t)y * w + x] = blur_pixel(in, w, h, x, y);
}

static void sobel(const uint8_t *in, uint8_t *out, int w, int h) {
    #pragma omp parallel for schedule(runtime)
    for (int y = 0; y < h; y++)
        for (int x = 0; x < w; x++)
            out[(size_t)y * w + x] = sobel_pixel(in, w, h, x, y);
}

/* Label for the CSV, e.g. "openmp_static", taken from the active schedule. */
static const char *version_label(void) {
#ifdef _OPENMP
    omp_sched_t kind;
    int chunk;
    omp_get_schedule(&kind, &chunk);
    switch ((int)((unsigned)kind & 0x7fffffffu)) { /* drop the "monotonic" flag bit */
        case omp_sched_static:  return "openmp_static";
        case omp_sched_dynamic: return "openmp_dynamic";
        case omp_sched_guided:  return "openmp_guided";
        default:                return "openmp_auto";
    }
#else
    return "openmp_disabled"; /* compiled without -fopenmp */
#endif
}

int main(int argc, char **argv) {
    if (argc < 3) {
        fprintf(stderr, "usage: %s input.pgm output.pgm [repeats]\n", argv[0]);
        return 1;
    }
    int reps = argc > 3 ? atoi(argv[3]) : 5;
    if (reps < 1) reps = 1;

#ifdef _OPENMP
    int threads = omp_get_max_threads();
#else
    int threads = 1;
#endif

    Image img = read_pgm(argv[1]);
    int w = img.width, h = img.height;
    uint8_t *blurred = (uint8_t *)xmalloc((size_t)w * h);
    uint8_t *edges = (uint8_t *)xmalloc((size_t)w * h);

    double best_blur = 0, best_sobel = 0, best_total = 1e30;
    for (int r = 0; r < reps; r++) {
        double t0 = now_ms();
        gaussian_blur(img.data, blurred, w, h);
        double t1 = now_ms();
        sobel(blurred, edges, w, h);
        double t2 = now_ms();
        if (t2 - t0 < best_total) {
            best_total = t2 - t0;
            best_blur = t1 - t0;
            best_sobel = t2 - t1;
        }
    }

    write_pgm(argv[2], edges, w, h);
    print_result(version_label(), threads, w, h, best_blur, best_sobel, 0.0);

    free(img.data);
    free(blurred);
    free(edges);
    return 0;
}

In [ ]:
%%writefile cuda.cu
/*
 * cuda.cu - the same two filters on an NVIDIA GPU, in two versions:
 *
 *   naive : one GPU thread per pixel; every thread reads its neighbours
 *           straight from (slow) global memory, so each pixel is read
 *           up to 25 times by different threads.
 *   tiled : each block first copies its tile of the image - plus a border
 *           ("halo") of neighbour pixels - into fast on-chip shared memory,
 *           waits for the whole block (__syncthreads), then computes from there.
 *
 * Threads are arranged as a 2D grid of 2D blocks (default 16 x 16 = 256
 * threads per block), so thread (x, y) handles pixel (x, y).
 *
 * usage: ./cuda_filter input.pgm output.pgm [naive|tiled] [block_size] [repeats]
 */
#include "common.h"
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err_ = (call);                                           \
        if (err_ != cudaSuccess) {                                           \
            fprintf(stderr, "CUDA error: %s (%s:%d)\n",                      \
                    cudaGetErrorString(err_), __FILE__, __LINE__);           \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

__constant__ int c_gauss[5] = {1, 4, 6, 4, 1};

/* ======================================================= version 1: naive */

__global__ void blur_naive(const uint8_t *in, uint8_t *out, int w, int h) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= h) return; /* the last blocks can hang off the image */

    int sum = 0;
    for (int dy = -2; dy <= 2; dy++)
        for (int dx = -2; dx <= 2; dx++)
            sum += c_gauss[dy + 2] * c_gauss[dx + 2] *
                   in[clampi(y + dy, 0, h - 1) * w + clampi(x + dx, 0, w - 1)];
    out[y * w + x] = (uint8_t)((sum + 128) >> 8);
}

__global__ void sobel_naive(const uint8_t *in, uint8_t *out, int w, int h) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= h) return;

    int xl = clampi(x - 1, 0, w - 1), xr = clampi(x + 1, 0, w - 1);
    int yu = clampi(y - 1, 0, h - 1), yd = clampi(y + 1, 0, h - 1);
    int p00 = in[yu * w + xl], p01 = in[yu * w + x], p02 = in[yu * w + xr];
    int p10 = in[y * w + xl],                         p12 = in[y * w + xr];
    int p20 = in[yd * w + xl], p21 = in[yd * w + x], p22 = in[yd * w + xr];

    int gx = (p02 + 2 * p12 + p22) - (p00 + 2 * p10 + p20);
    int gy = (p20 + 2 * p21 + p22) - (p00 + 2 * p01 + p02);
    out[y * w + x] = edge_value(gx, gy);
}

/* ================================================= version 2: shared memory */

/* All threads of the block cooperatively copy a (blockDim + 2R)^2 tile into
 * shared memory. The tile is bigger than the block, so each thread may copy
 * more than one pixel (hence the strided loops). */
__device__ void load_tile(const uint8_t *in, uint8_t *tile, int w, int h, int R) {
    int tw = blockDim.x + 2 * R, th = blockDim.y + 2 * R;
    int x0 = blockIdx.x * blockDim.x - R; /* image coords of the tile's top-left */
    int y0 = blockIdx.y * blockDim.y - R;
    for (int ty = threadIdx.y; ty < th; ty += blockDim.y)
        for (int tx = threadIdx.x; tx < tw; tx += blockDim.x)
            tile[ty * tw + tx] = in[clampi(y0 + ty, 0, h - 1) * w + clampi(x0 + tx, 0, w - 1)];
    __syncthreads(); /* nobody computes until the whole tile is loaded */
}

__global__ void blur_tiled(const uint8_t *in, uint8_t *out, int w, int h) {
    extern __shared__ uint8_t tile[]; /* size given at kernel launch */
    const int R = 2;
    load_tile(in, tile, w, h, R);

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= h) return; /* only after __syncthreads() */

    int tw = blockDim.x + 2 * R;
    const uint8_t *t = &tile[threadIdx.y * tw + threadIdx.x]; /* top-left of my 5x5 window */
    int sum = 0;
    for (int dy = 0; dy < 5; dy++)
        for (int dx = 0; dx < 5; dx++)
            sum += c_gauss[dy] * c_gauss[dx] * t[dy * tw + dx];
    out[y * w + x] = (uint8_t)((sum + 128) >> 8);
}

__global__ void sobel_tiled(const uint8_t *in, uint8_t *out, int w, int h) {
    extern __shared__ uint8_t tile[];
    const int R = 1;
    load_tile(in, tile, w, h, R);

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= w || y >= h) return;

    int tw = blockDim.x + 2 * R;
    const uint8_t *t = &tile[threadIdx.y * tw + threadIdx.x]; /* top-left of my 3x3 window */
    int p00 = t[0],          p01 = t[1],          p02 = t[2];
    int p10 = t[tw],                              p12 = t[tw + 2];
    int p20 = t[2 * tw],     p21 = t[2 * tw + 1], p22 = t[2 * tw + 2];

    int gx = (p02 + 2 * p12 + p22) - (p00 + 2 * p10 + p20);
    int gy = (p20 + 2 * p21 + p22) - (p00 + 2 * p01 + p02);
    out[y * w + x] = edge_value(gx, gy);
}

/* =================================================================== host */

int main(int argc, char **argv) {
    if (argc < 3) {
        fprintf(stderr, "usage: %s input.pgm output.pgm [naive|tiled] [block_size] [repeats]\n", argv[0]);
        return 1;
    }
    const char *mode = argc > 3 ? argv[3] : "tiled";
    int bs = argc > 4 ? atoi(argv[4]) : 16;
    int reps = argc > 5 ? atoi(argv[5]) : 5;
    bool tiled = strcmp(mode, "tiled") == 0;
    if (!tiled && strcmp(mode, "naive") != 0) die("mode must be 'naive' or 'tiled'");
    if (bs < 1 || bs > 32) die("block size must be 1..32 (at most 32x32 = 1024 threads per block)");
    if (reps < 1) reps = 1;

    Image img = read_pgm(argv[1]);
    int w = img.width, h = img.height;
    size_t n = (size_t)w * h;

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    fprintf(stderr, "GPU: %s (compute capability %d.%d, %d SMs)\n",
            prop.name, prop.major, prop.minor, prop.multiProcessorCount);

    uint8_t *d_in, *d_blurred, *d_edges;
    CUDA_CHECK(cudaMalloc(&d_in, n));
    CUDA_CHECK(cudaMalloc(&d_blurred, n));
    CUDA_CHECK(cudaMalloc(&d_edges, n));
    uint8_t *edges = (uint8_t *)xmalloc(n);

    dim3 block(bs, bs);
    dim3 grid((w + bs - 1) / bs, (h + bs - 1) / bs); /* enough blocks to cover the image */
    size_t smem_blur = (size_t)(bs + 4) * (bs + 4);  /* tile + 2-pixel halo */
    size_t smem_sobel = (size_t)(bs + 2) * (bs + 2); /* tile + 1-pixel halo */

    /* GPU-side timers: ev[0] copy-in ev[1] blur ev[2] sobel ev[3] copy-out ev[4] */
    cudaEvent_t ev[5];
    for (int i = 0; i < 5; i++) CUDA_CHECK(cudaEventCreate(&ev[i]));

    float best[4] = {0, 0, 0, 0};
    double best_total = 1e30;
    for (int r = 0; r < reps; r++) { /* the first run also pays one-time GPU setup */
        CUDA_CHECK(cudaEventRecord(ev[0]));
        CUDA_CHECK(cudaMemcpy(d_in, img.data, n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(ev[1]));
        if (tiled) blur_tiled<<<grid, block, smem_blur>>>(d_in, d_blurred, w, h);
        else       blur_naive<<<grid, block>>>(d_in, d_blurred, w, h);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(ev[2]));
        if (tiled) sobel_tiled<<<grid, block, smem_sobel>>>(d_blurred, d_edges, w, h);
        else       sobel_naive<<<grid, block>>>(d_blurred, d_edges, w, h);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(ev[3]));
        CUDA_CHECK(cudaMemcpy(edges, d_edges, n, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(ev[4]));
        CUDA_CHECK(cudaEventSynchronize(ev[4]));

        float t[4];
        for (int i = 0; i < 4; i++) CUDA_CHECK(cudaEventElapsedTime(&t[i], ev[i], ev[i + 1]));
        double total = t[0] + t[1] + t[2] + t[3];
        if (total < best_total) {
            best_total = total;
            for (int i = 0; i < 4; i++) best[i] = t[i];
        }
    }

    write_pgm(argv[2], edges, w, h);
    char label[32];
    snprintf(label, sizeof label, "cuda_%s", mode);
    /* blur kernel, sobel kernel, and the two copies counted as "transfer" */
    print_result(label, bs * bs, w, h, best[1], best[2], best[0] + best[3]);

    for (int i = 0; i < 5; i++) cudaEventDestroy(ev[i]);
    cudaFree(d_in);
    cudaFree(d_blurred);
    cudaFree(d_edges);
    free(img.data);
    free(edges);
    return 0;
}

In [ ]:
%%writefile Makefile
# make            -> serial + openmp (CPU versions)
# make cuda       -> cuda_filter (needs nvcc, e.g. on Google Colab)
# make clean

CC      ?= cc
CFLAGS  ?= -O2 -std=gnu11 -Wall -Wextra
NVCC    ?= nvcc
# sm_75 = Tesla T4 (Colab's free GPU); newer GPUs still run it.
NVFLAGS ?= -O2 -arch=sm_75

ifeq ($(shell uname),Darwin)
  # Apple clang needs Homebrew's OpenMP runtime: brew install libomp
  LIBOMP   := $(shell brew --prefix libomp 2>/dev/null)
  OMPFLAGS := -Xpreprocessor -fopenmp -I$(LIBOMP)/include
  OMPLIBS  := -L$(LIBOMP)/lib -lomp
else
  OMPFLAGS := -fopenmp
  OMPLIBS  := -fopenmp
endif

all: serial openmp

serial: serial.c common.h
	$(CC) $(CFLAGS) -o $@ serial.c -lm

openmp: openmp.c common.h
	$(CC) $(CFLAGS) $(OMPFLAGS) -o $@ openmp.c $(OMPLIBS) -lm

cuda: cuda_filter

cuda_filter: cuda.cu common.h
	$(NVCC) $(NVFLAGS) -o $@ cuda.cu

clean:
	rm -f serial openmp cuda_filter

.PHONY: all cuda clean

In [ ]:
%%writefile bench.sh
#!/usr/bin/env bash
# Runs every version on every image in images/ and saves the timings as CSV.
#
#   bash bench.sh cpu     serial + OpenMP (1, 2, 4, ... threads; static and dynamic)
#   bash bench.sh gpu     CUDA naive + tiled
#   bash bench.sh all     everything
#
# Optional settings:  REPS=5  THREADS="1 2 4 8"  BLOCKS="8 16 32"
set -euo pipefail
cd "$(dirname "$0")"

MODE="${1:-all}"
case "$MODE" in cpu|gpu|all) ;; *) echo "usage: bash bench.sh [cpu|gpu|all]" >&2; exit 1 ;; esac
REPS="${REPS:-5}"
BLOCKS="${BLOCKS:-16}"

# Thread counts: powers of two up to the number of cores, plus the core count.
NCPU=$(getconf _NPROCESSORS_ONLN)
if [ -z "${THREADS:-}" ]; then
  THREADS=""
  t=1
  while [ "$t" -lt "$NCPU" ]; do THREADS="$THREADS $t"; t=$((t * 2)); done
  THREADS="$THREADS $NCPU"
fi

shopt -s nullglob
images=(images/*.pgm)
if [ ${#images[@]} -eq 0 ]; then
  echo "No images in images/ - run: python3 tools.py prepare" >&2
  exit 1
fi

mkdir -p results out
CSV="results/$MODE.csv"
echo "version,threads,width,height,blur_ms,sobel_ms,transfer_ms,total_ms" > "$CSV"

for img in "${images[@]}"; do
  name=$(basename "$img" .pgm)
  echo "=== $name" >&2
  if [ "$MODE" != gpu ]; then
    ./serial "$img" "out/${name}_serial.pgm" "$REPS" | tee -a "$CSV"
    for sched in static dynamic; do
      for t in $THREADS; do
        OMP_SCHEDULE=$sched OMP_NUM_THREADS=$t \
          ./openmp "$img" "out/${name}_openmp.pgm" "$REPS" | tee -a "$CSV"
      done
    done
  fi
  if [ "$MODE" != cpu ]; then
    for v in naive tiled; do
      for b in $BLOCKS; do
        ./cuda_filter "$img" "out/${name}_cuda_${v}.pgm" "$v" "$b" "$REPS" | tee -a "$CSV"
      done
    done
  fi
done
echo "Saved $CSV" >&2

In [ ]:
%%writefile tools.py
#!/usr/bin/env python3
"""Image helpers for the project. Needs Pillow:  pip install pillow

  python3 tools.py prepare [--photo my.jpg]  make test images in images/ (HD, Full HD, 4K, 8K)
  python3 tools.py compare                   check every parallel output matches the serial one
  python3 tools.py topng                     convert images/ and out/ to PNG (png/) for slides
"""
import argparse
import glob
import os
import random
import sys

from PIL import Image, ImageChops, ImageDraw, ImageOps

SIZES = {"hd": (1280, 720), "fullhd": (1920, 1080), "4k": (3840, 2160), "8k": (7680, 4320)}


def synthetic(w, h, seed=7):
    """A test picture with lots of edges: shapes on a gradient, plus some noise."""
    rnd = random.Random(seed)
    img = Image.linear_gradient("L").resize((w, h))
    draw = ImageDraw.Draw(img)
    s = w / 1920
    for _ in range(80):
        x, y = rnd.randrange(w), rnd.randrange(h)
        r = int(rnd.uniform(20, 220) * s)
        fill = rnd.randrange(256)
        shape = rnd.choice(["ellipse", "rectangle", "line"])
        if shape == "line":
            end = (x + rnd.uniform(-500, 500) * s, y + rnd.uniform(-500, 500) * s)
            draw.line([(x, y), end], fill=fill, width=max(2, int(8 * s)))
        else:
            getattr(draw, shape)([x - r, y - r, x + r, y + r], fill=fill)
    noise = Image.effect_noise((w, h), 30)
    return Image.blend(img, noise, 0.12)


def prepare(args):
    os.makedirs("images", exist_ok=True)
    if args.photo:
        base = Image.open(args.photo).convert("L")
        print(f"Using {args.photo} ({base.width}x{base.height})")
    else:
        print("No --photo given, generating a synthetic test picture")
        base = synthetic(*SIZES["8k"])
    for name, size in SIZES.items():
        path = f"images/{name}.pgm"
        ImageOps.fit(base, size, Image.Resampling.LANCZOS).save(path)
        print(f"  {path}  {size[0]}x{size[1]}")


def compare(args):
    refs = sorted(glob.glob("out/*_serial.pgm"))
    if not refs:
        sys.exit("No out/*_serial.pgm yet - run: bash bench.sh cpu (or all)")
    all_ok = True
    for ref in refs:
        name = os.path.basename(ref)[: -len("_serial.pgm")]
        a = Image.open(ref)
        for other in sorted(glob.glob(f"out/{name}_*.pgm")):
            if other == ref:
                continue
            b = Image.open(other)
            if a.size != b.size:
                ok, msg = False, f"different size {b.size}"
            else:
                worst = ImageChops.difference(a, b).getextrema()[1]
                ok, msg = worst == 0, "identical" if worst == 0 else f"max pixel difference {worst}"
            all_ok &= ok
            print(f"{'OK  ' if ok else 'FAIL'} {other:34s} vs serial: {msg}")
    print("All outputs match the serial version." if all_ok else "Some outputs differ!")
    sys.exit(0 if all_ok else 1)


def topng(args):
    os.makedirs("png", exist_ok=True)
    for path in sorted(glob.glob("images/*.pgm") + glob.glob("out/*.pgm")):
        prefix = "input_" if path.startswith("images") else ""
        dest = f"png/{prefix}{os.path.basename(path)[:-4]}.png"
        Image.open(path).save(dest)
        print(dest)
    # Side-by-side "before | after" picture for the slides.
    for name in SIZES:
        src, edges = f"images/{name}.pgm", f"out/{name}_serial.pgm"
        if os.path.exists(src) and os.path.exists(edges):
            a, b = Image.open(src), Image.open(edges)
            pair = Image.new("L", (a.width * 2 + 20, a.height), 255)
            pair.paste(a, (0, 0))
            pair.paste(b, (a.width + 20, 0))
            pair.save(f"png/before_after_{name}.png")
            print(f"png/before_after_{name}.png")


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    sp = sub.add_parser("prepare", help="create the test images")
    sp.add_argument("--photo", help="use this photo instead of a synthetic picture")
    sub.add_parser("compare", help="verify outputs against the serial version")
    sub.add_parser("topng", help="convert results to PNG")
    args = p.parse_args()
    {"prepare": prepare, "compare": compare, "topng": topng}[args.cmd](args)

In [ ]:
%%writefile plot.py
#!/usr/bin/env python3
"""Turn benchmark CSVs into charts + a summary table for the slides.
Needs matplotlib:  pip install matplotlib

  python3 plot.py                              uses every results/*.csv
  python3 plot.py results/cpu.csv results/gpu.csv

If the same measurement appears in several files, the later file wins - so you
can combine OpenMP numbers from your laptop with CUDA numbers from Colab.
Writes charts/*.png and charts/summary.md
"""
import csv
import glob
import os
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

SIZE_NAMES = {(1280, 720): "HD", (1920, 1080): "Full HD", (3840, 2160): "4K", (7680, 4320): "8K"}
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # fixed order: blue, orange, aqua, yellow
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 150, "font.size": 11, "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "text.color": INK,
})


def load(paths):
    data = {}
    for path in paths:
        with open(path) as f:
            for r in csv.DictReader(f):
                for k in ("threads", "width", "height"):
                    r[k] = int(r[k])
                for k in ("blur_ms", "sobel_ms", "transfer_ms", "total_ms"):
                    r[k] = float(r[k])
                data[(r["version"], r["threads"], r["width"], r["height"])] = r
    return list(data.values())


def size_name(wh):
    return SIZE_NAMES.get(wh, f"{wh[0]}x{wh[1]}")


def pick(rows, wh, prefix):
    """Fastest row for this image size whose version starts with prefix."""
    cands = [r for r in rows if (r["width"], r["height"]) == wh and r["version"].startswith(prefix)]
    return min(cands, key=lambda r: r["total_ms"]) if cands else None


def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"charts/{name}", bbox_inches="tight")
    plt.close(fig)
    print(f"charts/{name}")


def chart_versions(rows, sizes):
    """Grouped bars: total time of each version per image size (log scale)."""
    series = [("Serial", "serial"), ("OpenMP (best)", "openmp"),
              ("CUDA naive", "cuda_naive"), ("CUDA shared mem", "cuda_tiled")]
    series = [s for s in series if any(pick(rows, wh, s[1]) for wh in sizes)]
    fig, ax = plt.subplots(figsize=(10, 5.5))
    width = 0.8 / len(series)
    for i, (label, prefix) in enumerate(series):
        xs, ys = [], []
        for j, wh in enumerate(sizes):
            r = pick(rows, wh, prefix)
            if r:
                xs.append(j + (i - (len(series) - 1) / 2) * width)
                ys.append(r["total_ms"])
        ax.bar(xs, ys, width * 0.92, label=label, color=COLORS[i])
        if prefix != "serial":  # speedup over serial on top of each bar
            for x, y, wh in zip(xs, ys, sizes):
                s = pick(rows, wh, "serial")
                if s:
                    ax.text(x, y * 1.08, f"{s['total_ms'] / y:.0f}x", ha="center", fontsize=8, color=MUTED)
    ax.set_yscale("log")
    ax.set_xticks(range(len(sizes)), [size_name(wh) for wh in sizes])
    ax.set_ylabel("Total time (ms, log scale) - lower is better")
    ax.set_title("Blur + edge detection: time per image")
    ax.legend(ncol=len(series), loc="upper left")
    ax.grid(axis="x", visible=False)
    save(fig, "1_time_by_version.png")


def openmp_curve(rows, wh, sched):
    serial = pick(rows, wh, "serial")
    pts = sorted((r["threads"], r["total_ms"]) for r in rows
                 if r["version"] == f"openmp_{sched}" and (r["width"], r["height"]) == wh)
    if not serial or not pts:
        return [], []
    return [t for t, _ in pts], [serial["total_ms"] / ms for _, ms in pts]


def chart_openmp(rows, sizes):
    """Speedup and efficiency vs threads (static schedule), one line per image size."""
    curves = [(wh, *openmp_curve(rows, wh, "static")) for wh in sizes]
    curves = [c for c in curves if c[1]]
    if not curves:
        return
    max_t = max(max(c[1]) for c in curves)
    for kind in ("speedup", "efficiency"):
        fig, ax = plt.subplots(figsize=(8, 5.5))
        if kind == "speedup":
            ax.plot([1, max_t], [1, max_t], "--", color=MUTED, lw=1.2, label="Ideal (linear)")
        else:
            ax.axhline(100, ls="--", color=MUTED, lw=1.2, label="Ideal (100%)")
        for i, (wh, ts, sp) in enumerate(curves):
            ys = sp if kind == "speedup" else [100 * s / t for s, t in zip(sp, ts)]
            ax.plot(ts, ys, "-o", lw=2, ms=6, color=COLORS[i], label=size_name(wh))
        ax.set_xticks(curves[0][1])
        ax.set_xlabel("OpenMP threads")
        if kind == "speedup":
            ax.set_ylabel("Speedup over serial (x)")
            ax.set_title("OpenMP speedup (static schedule)")
        else:
            ax.set_ylabel("Efficiency = speedup / threads (%)")
            ax.set_ylim(0, 115)
            ax.set_title("OpenMP efficiency (static schedule)")
        ax.legend()
        save(fig, f"{2 if kind == 'speedup' else 3}_openmp_{kind}.png")


def chart_schedule(rows, sizes):
    """static vs dynamic scheduling on the largest image."""
    wh = sizes[-1]
    lines = [(s, *openmp_curve(rows, wh, s)) for s in ("static", "dynamic")]
    lines = [line for line in lines if line[1]]
    if len(lines) < 2:
        return
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for i, (sched, ts, sp) in enumerate(lines):
        ax.plot(ts, sp, "-o", lw=2, ms=6, color=COLORS[i], label=f"schedule({sched})")
    ax.set_xticks(lines[0][1])
    ax.set_xlabel("OpenMP threads")
    ax.set_ylabel("Speedup over serial (x)")
    ax.set_title(f"Static vs dynamic scheduling ({size_name(wh)} image)")
    ax.legend()
    save(fig, "4_openmp_schedule.png")


def chart_gpu(rows, sizes):
    """Where the GPU time goes: copies vs the two kernels."""
    bars = []
    for wh in sizes:
        for prefix, label in (("cuda_naive", "naive"), ("cuda_tiled", "shared")):
            r = pick(rows, wh, prefix)
            if r:
                bars.append((f"{size_name(wh)}\n{label}", r))
    if not bars:
        return
    fig, ax = plt.subplots(figsize=(10, 5.5))
    xs = range(len(bars))
    bottom = [0.0] * len(bars)
    for i, (key, label) in enumerate([("transfer_ms", "CPU <-> GPU copies"),
                                      ("blur_ms", "Blur kernel"), ("sobel_ms", "Sobel kernel")]):
        vals = [r[key] for _, r in bars]
        ax.bar(xs, vals, 0.6, bottom=bottom, color=COLORS[i], label=label, edgecolor="white", linewidth=1.5)
        bottom = [b + v for b, v in zip(bottom, vals)]
    for x, (_, r) in zip(xs, bars):
        pct = 100 * r["transfer_ms"] / r["total_ms"]
        ax.text(x, r["total_ms"] * 1.02, f"{pct:.0f}% copying", ha="center", fontsize=8, color=MUTED)
    ax.set_xticks(list(xs), [b[0] for b in bars], fontsize=9)
    ax.set_ylabel("Time (ms)")
    ax.set_title("GPU time breakdown: copying data vs computing")
    ax.legend(loc="upper left")
    ax.grid(axis="x", visible=False)
    save(fig, "5_gpu_breakdown.png")


def summary(rows, sizes):
    lines = ["| Image | Serial (ms) | Best OpenMP (ms) | OpenMP speedup | Efficiency | "
             "Serial fraction f (Karp-Flatt) | CUDA kernels only (ms) | CUDA incl. copies (ms) | "
             "GPU speedup (kernels / incl. copies) |",
             "|---|---|---|---|---|---|---|---|---|"]
    for wh in sizes:
        s, o = pick(rows, wh, "serial"), pick(rows, wh, "openmp")
        g = pick(rows, wh, "cuda")
        cells = [size_name(wh), f"{s['total_ms']:.1f}" if s else "-"]
        if s and o:
            sp, p = s["total_ms"] / o["total_ms"], o["threads"]
            # Karp-Flatt metric: experimentally measured serial fraction (Amdahl's f)
            f = (1 / sp - 1 / p) / (1 - 1 / p) if p > 1 else float("nan")
            cells += [f"{o['total_ms']:.1f} ({p} threads)", f"{sp:.1f}x", f"{100 * sp / p:.0f}%", f"{f:.3f}"]
        else:
            cells += ["-"] * 4
        if g:
            kern = g["blur_ms"] + g["sobel_ms"]
            cells += [f"{kern:.2f}", f"{g['total_ms']:.2f}"]
            cells.append(f"{s['total_ms'] / kern:.0f}x / {s['total_ms'] / g['total_ms']:.0f}x" if s else "-")
        else:
            cells += ["-"] * 3
        lines.append("| " + " | ".join(cells) + " |")
    text = "\n".join(lines) + "\n"
    with open("charts/summary.md", "w") as f:
        f.write(text)
    print("charts/summary.md\n\n" + text)


def main():
    paths = sys.argv[1:] or sorted(glob.glob("results/*.csv"))
    if not paths:
        sys.exit("No results/*.csv yet - run bench.sh first")
    rows = load(paths)
    sizes = sorted({(r["width"], r["height"]) for r in rows}, key=lambda wh: wh[0] * wh[1])
    os.makedirs("charts", exist_ok=True)
    chart_versions(rows, sizes)
    chart_openmp(rows, sizes)
    chart_schedule(rows, sizes)
    chart_gpu(rows, sizes)
    summary(rows, sizes)


if __name__ == "__main__":
    main()

## 3. Make the test images
Set `USE_MY_PHOTO = True` to upload your own photo (a real photo looks better on
the slides). Otherwise a synthetic test picture is generated.

In [ ]:
import subprocess
USE_MY_PHOTO = False

args = []
if USE_MY_PHOTO:
    from google.colab import files
    args = ["--photo", list(files.upload())[0]]
subprocess.run(["python3", "tools.py", "prepare", *args], check=True)

## 4. Compile all three versions

In [ ]:
!make all cuda

## 5. Benchmark everything
Serial, OpenMP (static + dynamic), CUDA naive and CUDA shared-memory, on every image size.

In [ ]:
!bash bench.sh all

## 6. Check correctness
Every parallel output must be pixel-identical to the serial output.

In [ ]:
!python3 tools.py compare

## 7. Charts
Tip: Colab only has 2 CPU cores, so the OpenMP scaling charts look better when
run on your laptop (`bash bench.sh cpu`). The CPU-vs-GPU comparison below is fair
because everything ran on this same machine.

In [ ]:
!python3 plot.py results/all.csv
from IPython.display import Image, Markdown, display
import glob
for p in sorted(glob.glob("charts/*.png")):
    display(Image(p, width=800))
display(Markdown(open("charts/summary.md").read()))

## 8. Before / after

In [ ]:
!python3 tools.py topng > /dev/null
display(Image("png/before_after_fullhd.png", width=1000))

## 9. Download results, charts and images for your slides

In [ ]:
!zip -qr results.zip results charts png/before_after_*.png
from google.colab import files
files.download("results.zip")

## 10. Live demo (for the presentation)
Run this one cell in front of the audience (~10 seconds). It needs sections 1-5
to have run first: do *Run all* before the presentation starts.

In [ ]:
import subprocess
from PIL import Image as PILImage

def run(*cmd):
    out = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return out.stdout.strip().splitlines()[-1].split(",")

gpu_name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu_name}    Image: 8K, 7680 x 4320 = 33.2 million pixels\n")
for mode in ("naive", "tiled"):
    r = run("./cuda_filter", "images/8k.pgm", f"out/demo_{mode}.pgm", mode, "16", "5")
    blur, sobel, copy, total = map(float, r[4:8])
    name = "naive (global memory)" if mode == "naive" else "shared memory (tiled)"
    print(f"CUDA {name:22s} blur {blur:5.2f} ms | sobel {sobel:5.2f} ms | "
          f"copies {copy:5.2f} ms | total {total:5.2f} ms  ({100 * copy / total:.0f}% copying)")

run("./serial", "images/8k.pgm", "out/demo_serial.pgm", "1")
same = all(open("out/demo_serial.pgm", "rb").read() == open(f"out/demo_{m}.pgm", "rb").read()
           for m in ("naive", "tiled"))
print("\nGPU output identical to the serial CPU output:", "YES" if same else "NO")
display(PILImage.open("out/demo_tiled.pgm").resize((1600, 900)))